# 2025

In [8]:
import pandas as pd
import numpy as np
from pathlib import Path
from openpyxl import load_workbook
from openpyxl.styles import PatternFill, Font, Alignment, Border, Side
from openpyxl.utils import get_column_letter

# 노트북 기준 상대 경로
RAW_2025   = Path("../data/1_raw/DATA_2025년 국민생활체육조사.xlsx")
OUTPUT_DIR = Path("../data/3_variable_define")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

YEAR = 2025
OUTPUT_FILE = OUTPUT_DIR / f"{YEAR}_variable_define.xlsx"
print("경로 설정 완료")

# ── 헤더 5행 로드
hdr = pd.read_excel(RAW_2025, header=None, nrows=5)

# row0: 섹션명 (병합셀 → ffill)
sec_row  = hdr.iloc[0].ffill()
# row1: 주 설명
desc1    = hdr.iloc[1]
# row2: 보조 설명 (지역→시도/시군구/읍면동 등)
desc2    = hdr.iloc[2]
# row4: 변수코드
code_row = hdr.iloc[4]

n_cols = len(code_row)
print(f"총 컬럼 수: {n_cols}")
print(f"섹션 목록: {sec_row.dropna().unique().tolist()}")

def build_description(i, desc1, desc2):
    """
    row1(주설명) + row2(보조설명) 조합
    - row1이 있으면 row1 사용
    - row1이 NaN이고 row2가 있으면 → 직전 row1 + ' | ' + row2
    - 둘 다 NaN이면 → 직전 row1 (순번 항목)
    """
    d1 = str(desc1.iloc[i]).strip() if not pd.isna(desc1.iloc[i]) else ""
    d2 = str(desc2.iloc[i]).strip() if not pd.isna(desc2.iloc[i]) else ""

    # 개행문자 정리
    d1 = d1.replace("\n", " ").strip()
    d2 = d2.replace("\n", " ").strip()

    if d1 and d2:
        return f"{d1} | {d2}"
    elif d1:
        return d1
    elif d2:
        # row1은 NaN → ffill로 부모 문항 찾기
        parent = ""
        for j in range(i - 1, -1, -1):
            pv = str(desc1.iloc[j]).strip() if not pd.isna(desc1.iloc[j]) else ""
            if pv:
                parent = pv.replace("\n", " ").strip()
                break
        return f"{parent} | {d2}" if parent else d2
    else:
        # 둘 다 NaN → 부모 문항명만
        for j in range(i - 1, -1, -1):
            pv = str(desc1.iloc[j]).strip() if not pd.isna(desc1.iloc[j]) else ""
            if pv:
                return pv.replace("\n", " ").strip()
        return ""

# 테스트 (처음 15개)
for i in range(15):
    code = code_row.iloc[i]
    sec  = sec_row.iloc[i]
    desc = build_description(i, desc1, desc2)
    print(f"  [{i:3d}] {str(code):<12}  섹션={str(sec)[:20]:<20}  설명={desc[:50]}")
    
    
    
# ── 실제 데이터 로드 (row4=변수코드, row5~=데이터)
df = pd.read_excel(RAW_2025, header=4)

# 첫 행이 변수코드 잔류 시 제거
if str(df.iloc[0, 0]).strip() == str(code_row.iloc[0]).strip():
    df = df.iloc[1:].reset_index(drop=True)

print(f"데이터: {df.shape[0]:,}행 × {df.shape[1]}열")
df.head(2)
MAX_UNIQ_DISPLAY = 50   # 유니크값 최대 표시 개수

rows = []
for i, code in enumerate(code_row):
    if pd.isna(code):
        continue
    code_str = str(code).strip()
    sec_str  = str(sec_row.iloc[i]).strip() if not pd.isna(sec_row.iloc[i]) else ""
    desc_str = build_description(i, desc1, desc2)

    # 실제 컬럼 존재 여부
    col_data = df[code_str] if code_str in df.columns else pd.Series(dtype=object)

    # 데이터 타입
    if pd.api.types.is_numeric_dtype(col_data):
        dtype = "숫자"
    else:
        dtype = "텍스트/혼합"

    # 유니크값
    uniq_vals = col_data.dropna().unique()
    try:
        uniq_sorted = sorted(uniq_vals, key=lambda x: (isinstance(x, str), x))
    except TypeError:
        uniq_sorted = list(uniq_vals)

    n_uniq = len(uniq_sorted)
    uniq_display = uniq_sorted[:MAX_UNIQ_DISPLAY]
    uniq_str = ", ".join(str(v) for v in uniq_display)
    if n_uniq > MAX_UNIQ_DISPLAY:
        uniq_str += "..."

    rows.append({
        "섹션":     sec_str,
        "변수코드": code_str,
        "변수설명": desc_str,
        "데이터타입": dtype,
        "유니크값수": n_uniq,
        "유니크값":  uniq_str,
        "표준컬럼명": "",  # 사용자 입력
        "제거여부":   "",  # Y 입력 시 제거
        "비고":       "",
    })

df_define = pd.DataFrame(rows)
print(f"변수정의서: {len(df_define)}행")
df_define.head(10)
# ── 섹션별 색상
SEC_COLORS = {
    "관리사항 및 응답자 정보": "D9E1F2",
    "Ⅰ. 건강과 체력상태":     "E2EFDA",
    "Ⅱ. 현재 체육활동 참여 현황": "FFF2CC",
    "Ⅲ. 체육활동 및 여건":    "FCE4D6",
    "Ⅴ. 개인 관련 사항":      "EDEDED",
}
DEFAULT_COLOR = "FFFFFF"

HDR_FILL = PatternFill("solid", fgColor="1F4E79")
HDR_FONT = Font(name="Arial", bold=True, color="FFFFFF", size=10)
BODY_FONT = Font(name="Arial", size=9)
thin = Side(style="thin", color="CCCCCC")
BORDER = Border(left=thin, right=thin, top=thin, bottom=thin)

# ── pandas로 먼저 저장
df_define.to_excel(OUTPUT_FILE, index=False)

# ── openpyxl로 서식 적용
wb = load_workbook(OUTPUT_FILE)
ws = wb.active
ws.title = f"{YEAR}_variable_define"
ws.freeze_panes = "A2"

# 헤더 서식
for cell in ws[1]:
    cell.fill = HDR_FILL
    cell.font = HDR_FONT
    cell.alignment = Alignment(horizontal="center", vertical="center")
    cell.border = BORDER
ws.row_dimensions[1].height = 20

# 데이터 서식
for r in range(2, ws.max_row + 1):
    sec_val = str(ws.cell(r, 1).value or "")
    fill_color = SEC_COLORS.get(sec_val, DEFAULT_COLOR)
    row_fill = PatternFill("solid", fgColor=fill_color)

    for c in range(1, ws.max_column + 1):
        cell = ws.cell(r, c)
        cell.fill = row_fill
        cell.font = BODY_FONT
        cell.alignment = Alignment(vertical="center", wrap_text=(c in (3, 6)))
        cell.border = BORDER
    ws.row_dimensions[r].height = 30

# 컬럼 너비
col_widths = [22, 14, 45, 12, 10, 60, 16, 10, 20]
for i, w in enumerate(col_widths, 1):
    ws.column_dimensions[get_column_letter(i)].width = w

# 안내 시트 추가
ws_guide = wb.create_sheet("안내")
guide_data = [
    ("컬럼명",    "설명"),
    ("섹션",      "변수 그룹 (색상 구분)"),
    ("변수코드",  "원본 엑셀의 컬럼명"),
    ("변수설명",  "원본 헤더에서 추출한 한글 설명"),
    ("데이터타입","숫자 / 텍스트·혼합"),
    ("유니크값수","해당 컬럼의 고유값 개수"),
    ("유니크값",  "실제 들어있는 값들 (최대 50개)"),
    ("표준컬럼명","★ 여기에 원하는 표준 변수명 입력 (비워두면 제거 대상)"),
    ("제거여부",  "★ Y 입력 시 제거 대상 (표준컬럼명이 있으면 무시됨)"),
    ("비고",      "메모"),
]
for row in guide_data:
    ws_guide.append(row)
ws_guide.column_dimensions["A"].width = 14
ws_guide.column_dimensions["B"].width = 50

wb.save(OUTPUT_FILE)
print(f"✅ 저장 완료: {OUTPUT_FILE.resolve()}")
print(f"   총 {len(df_define)}개 변수")
print(f"\n섹션별 변수 수:")
print(df_define["섹션"].value_counts().to_string())


경로 설정 완료
총 컬럼 수: 228
섹션 목록: ['관리사항 및 응답자 정보', 'Ⅰ. 건강과 체력상태', 'Ⅱ. 현재 체육활동 참여 현황', 'Ⅲ. 체육활동 및 여건', 'Ⅴ. 개인 관련 사항']
  [  0] ID            섹션=관리사항 및 응답자 정보         설명=ID
  [  1] 일련번호          섹션=관리사항 및 응답자 정보         설명=조사구 일련번호
  [  2] CODE1         섹션=관리사항 및 응답자 정보         설명=조사구번호
  [  3] CODE2         섹션=관리사항 및 응답자 정보         설명=가구번호
  [  4] APT           섹션=관리사항 및 응답자 정보         설명=주거유형
  [  5] CODE3         섹션=관리사항 및 응답자 정보         설명=지역 | 시도
  [  6] CODE4         섹션=관리사항 및 응답자 정보         설명=지역 | 시군구
  [  7] CODE5         섹션=관리사항 및 응답자 정보         설명=지역 | 읍면동
  [  8] CODE6         섹션=관리사항 및 응답자 정보         설명=읍면동(동부/읍면부)
  [  9] SEX           섹션=관리사항 및 응답자 정보         설명=성별
  [ 10] YEAR          섹션=관리사항 및 응답자 정보         설명=출생연도
  [ 11] MON           섹션=관리사항 및 응답자 정보         설명=출생월
  [ 12] AGE           섹션=관리사항 및 응답자 정보         설명=연령
  [ 13] Q1            섹션=Ⅰ. 건강과 체력상태           설명=문1. 본인의 건강상태에 대한 인식
  [ 14] Q2            섹션=Ⅰ. 건강과 체력상태           설명=문2. 건강 유지 위해 중요한 것
데이터: 9,000행 × 228열

In [9]:
from pathlib import Path
import pandas as pd
import numpy as np
from openpyxl import load_workbook
from openpyxl.styles import PatternFill, Font, Alignment, Border, Side
from openpyxl.utils import get_column_letter

# ── 경로
RAW_2025     = Path("../data/1_raw/DATA_2025년 국민생활체육조사.xlsx")
VAR_DEF_2025 = Path("../data/3_variable_define/2025_variable_define.xlsx")
OUTPUT_DIR   = Path("../data/2_codebook")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_FILE  = OUTPUT_DIR / "2025_codebook.xlsx"

# ── 데이터 로드
df_def = pd.read_excel(VAR_DEF_2025, sheet_name="2025_variable_define")
df_raw = pd.read_excel(RAW_2025, header=4)
if str(df_raw.iloc[0, 0]).strip() == "ID":
    df_raw = df_raw.iloc[1:].reset_index(drop=True)
print(f"변수정의서: {len(df_def)}개 / 원본데이터: {df_raw.shape[0]:,}행")

# ── 변수 유형 분류
SPORT_CODE_PATTERNS = (
    "Q6_", "Q8_1_1", "Q8_2_1", "Q8_3_1",
    "Q18_1", "Q18_2", "Q22_1", "Q22_2",
    "Q24_1_", "Q28_1_", "Q31_1_", "Q31_4_A",
)
MGMT_VARS       = {"ID", "일련번호", "CODE1", "CODE2", "APT"}
DEMO_CONTINUOUS = {"YEAR", "AGE", "MON"}
CONTINUOUS_KW   = ("비용", "소득", "기간", "일수", "시간", "분")

def classify_var(code, desc, dtype, n_uniq):
    if code.endswith(("t5","t6","t7","t8","t9","t10","t11","t12")) or "기타 응답" in desc:
        return "기타응답"
    if code in MGMT_VARS:
        return "관리변수"
    if code.startswith("CODE"):
        return "지역_텍스트" if dtype == "텍스트/혼합" else "관리변수"
    if code in DEMO_CONTINUOUS:
        return "인구통계_연속"
    if any(code.startswith(p) for p in SPORT_CODE_PATTERNS):
        return "종목·시설코드"
    if dtype == "텍스트/혼합":
        return "범주형_텍스트"
    if n_uniq == 0:
        return "응답없음"
    if n_uniq > 30 or any(kw in desc for kw in CONTINUOUS_KW):
        return "연속형"
    return "범주형_코드"

df_def["변수유형"] = df_def.apply(
    lambda r: classify_var(str(r["변수코드"]), str(r["변수설명"]),
                           str(r["데이터타입"]), int(r["유니크값수"])), axis=1
)
print("\n변수 유형 분포:")
print(df_def["변수유형"].value_counts().to_string())

# ── 코드북 Long Format 생성
MAX_VALUES = 300
rows = []

for _, var_row in df_def.iterrows():
    code   = str(var_row["변수코드"]).strip()
    desc   = str(var_row["변수설명"]).strip()
    sec    = str(var_row["섹션"]).strip()
    vtype  = str(var_row["변수유형"]).strip()
    n_uniq = int(var_row["유니크값수"])
    uniq_raw = str(var_row["유니크값"]).strip()
    base = dict(섹션=sec, 변수코드=code, 변수설명=desc, 변수유형=vtype)

    if vtype in ("연속형", "관리변수", "인구통계_연속", "응답없음"):
        rows.append({**base, "코드값": "(연속형/자유값)", "코드레이블": "",
                     "비고": f"유니크값 {n_uniq}개"})
        continue
    if vtype == "기타응답":
        rows.append({**base, "코드값": "(주관식)", "코드레이블": "", "비고": "텍스트 자유응답"})
        continue

    raw_vals = [v.strip() for v in uniq_raw.rstrip("...").split(",") if v.strip()] if uniq_raw not in ("nan","") else []
    if not raw_vals:
        rows.append({**base, "코드값": "(없음)", "코드레이블": "", "비고": ""})
        continue

    if vtype in ("범주형_텍스트", "지역_텍스트"):
        for val in raw_vals[:MAX_VALUES]:
            rows.append({**base, "코드값": val, "코드레이블": val,
                         "비고": f"전체 {n_uniq}개 중 {MAX_VALUES}개만 표시" if n_uniq > MAX_VALUES else ""})
        continue

    # 범주형_코드 / 종목·시설코드
    parsed = []
    for v in raw_vals:
        try:    parsed.append(int(float(v)))
        except: parsed.append(v)
    for val in sorted(set(parsed), key=lambda x: (isinstance(x, str), x)):
        rows.append({**base, "코드값": val, "코드레이블": "",
                     "비고": "종목/시설 분류코드" if vtype == "종목·시설코드" else ""})

df_codebook = pd.DataFrame(rows)
print(f"\n코드북 총 행수: {len(df_codebook):,}")

# ── Excel 저장
VTYPE_COLORS = {
    "범주형_코드":     "FFF2CC",
    "종목·시설코드":   "D9E1F2",
    "범주형_텍스트":   "E2EFDA",
    "지역_텍스트":     "E2EFDA",
    "연속형":          "EFEFEF",
    "인구통계_연속":   "EFEFEF",
    "관리변수":        "F3F3F3",
    "기타응답":        "FCE4D6",
    "응답없음":        "F4CCCC",
}
HDR_FILL  = PatternFill("solid", fgColor="1F4E79")
HDR_FONT  = Font(name="Arial", bold=True, color="FFFFFF", size=10)
BODY_FONT = Font(name="Arial", size=9)
INPUT_FONT = Font(name="Arial", size=9, color="1F4E79", bold=True)
thin   = Side(style="thin", color="CCCCCC")
BORDER = Border(left=thin, right=thin, top=thin, bottom=thin)

df_codebook.to_excel(OUTPUT_FILE, index=False, sheet_name="코드정의")
wb = load_workbook(OUTPUT_FILE)
ws = wb["코드정의"]
ws.freeze_panes = "A2"

for cell in ws[1]:
    cell.fill = HDR_FILL; cell.font = HDR_FONT
    cell.alignment = Alignment(horizontal="center", vertical="center")
    cell.border = BORDER
ws.row_dimensions[1].height = 22

# "코드레이블" 컬럼 인덱스 찾기
header_map = {cell.value: cell.column for cell in ws[1]}
col_label = header_map.get("코드레이블", 6)

for r in range(2, ws.max_row + 1):
    vtype = str(ws.cell(r, 4).value or "")
    row_fill = PatternFill("solid", fgColor=VTYPE_COLORS.get(vtype, "FFFFFF"))
    for c in range(1, ws.max_column + 1):
        cell = ws.cell(r, c)
        cell.fill = row_fill
        cell.font = INPUT_FONT if c == col_label and vtype in ("범주형_코드", "종목·시설코드") else BODY_FONT
        cell.alignment = Alignment(vertical="center", wrap_text=(c in (3,)))
        cell.border = BORDER
    ws.row_dimensions[r].height = 18

col_widths = [24, 14, 50, 16, 12, 28, 30]
for i, w in enumerate(col_widths, 1):
    if i <= ws.max_column:
        ws.column_dimensions[get_column_letter(i)].width = w

# 범례 시트
ws_leg = wb.create_sheet("범례_안내")
legend = [
    ("변수유형",        "설명",                         "처리 방법"),
    ("범주형_코드",     "숫자코드 → 한글레이블 필요",   "★ 코드레이블 컬럼에 직접 입력"),
    ("종목·시설코드",   "3자리 분류코드 (101, 401 등)", "★ 종목코드표 참조하여 입력"),
    ("범주형_텍스트",   "텍스트 값이 그대로 레이블",    "자동 입력 (수정 가능)"),
    ("지역_텍스트",     "시도·시군구·읍면동",           "자동 입력 (수정 가능)"),
    ("연속형",          "자유 수치값 (비용·소득 등)",   "입력 불필요"),
    ("인구통계_연속",   "출생연도·연령·출생월",         "입력 불필요"),
    ("관리변수",        "ID·조사구번호 등",             "입력 불필요"),
    ("기타응답",        "주관식 텍스트 응답",           "입력 불필요"),
    ("응답없음",        "해당 응답 없는 변수",          "제거 검토"),
]
leg_hdr_fill = PatternFill("solid", fgColor="1F4E79")
for i, row_data in enumerate(legend, 1):
    for j, val in enumerate(row_data, 1):
        c = ws_leg.cell(i, j, val)
        c.border = BORDER
        if i == 1:
            c.fill = leg_hdr_fill; c.font = HDR_FONT
            c.alignment = Alignment(horizontal="center", vertical="center")
        else:
            c.fill = PatternFill("solid", fgColor=VTYPE_COLORS.get(row_data[0], "FFFFFF"))
            c.font = BODY_FONT; c.alignment = Alignment(vertical="center")
    ws_leg.row_dimensions[i].height = 20
for i, w in enumerate([16, 30, 30], 1):
    ws_leg.column_dimensions[get_column_letter(i)].width = w

wb.save(OUTPUT_FILE)
print(f"\n✅ 저장 완료: {OUTPUT_FILE.resolve()}")

변수정의서: 228개 / 원본데이터: 9,000행

변수 유형 분포:
변수유형
범주형_코드     96
종목·시설코드    65
기타응답       26
연속형        20
응답없음        7
관리변수        5
지역_텍스트      4
인구통계_연속     3
범주형_텍스트     2

코드북 총 행수: 1,821

✅ 저장 완료: C:\Users\hanbi\OneDrive\Desktop\korean-sports-survey-ml\data\2_codebook\2025_codebook.xlsx


In [10]:
from pathlib import Path
import pandas as pd
import numpy as np

# ── 경로
RAW_2025     = Path("../data/1_raw/DATA_2025년 국민생활체육조사.xlsx")
VAR_DEF_2025 = Path("../data/3_variable_define/2025_variable_define.xlsx")
CODEBOOK     = Path("../data/2_codebook/2025_codebook.xlsx")
OUTPUT_DIR   = Path("../data/5_processed")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# ══════════════════════════════════════════
# 1. 원본 데이터 로드
# ══════════════════════════════════════════
df = pd.read_excel(RAW_2025, header=4)
if str(df.iloc[0, 0]).strip() == "ID":
    df = df.iloc[1:].reset_index(drop=True)
print(f"[원본] {df.shape[0]:,}행 × {df.shape[1]}열")

# ══════════════════════════════════════════
# 2. 변수정의서 로드 → 제거 / 컬럼명 변경
# ══════════════════════════════════════════
df_def = pd.read_excel(VAR_DEF_2025, sheet_name="2025_variable_define")

# 제거 대상 (제거여부 == 'Y')
drop_cols = set(
    df_def.loc[df_def["제거여부"].astype(str).str.strip().str.upper() == "Y", "변수코드"]
)

# 컬럼명 변경 (표준컬럼명이 채워진 것만)
rename_map = {}
for _, row in df_def.iterrows():
    std = str(row.get("표준컬럼명", "")).strip()
    orig = str(row["변수코드"]).strip()
    if std and std not in ("nan", "") and std != orig:
        rename_map[orig] = std

print(f"\n[변수정의서]")
print(f"  제거 대상: {len(drop_cols)}개  → {sorted(drop_cols)}")
print(f"  컬럼명 변경: {len(rename_map)}개  → {rename_map}")

# ── 제거
df = df.drop(columns=[c for c in drop_cols if c in df.columns], errors="ignore")

# ── 컬럼명 변경
df = df.rename(columns=rename_map)
print(f"\n[컬럼처리 후] {df.shape[1]}열")

# ══════════════════════════════════════════
# 3. 코드북 로드 → 코드값 → 한글 레이블 매핑
# ══════════════════════════════════════════
df_cb = pd.read_excel(CODEBOOK, sheet_name="코드정의")

# 컬럼명이 rename된 경우를 위해 역방향 맵도 생성
rev_rename = {v: k for k, v in rename_map.items()}  # 표준명 → 원본명

# {현재컬럼명: {코드값: 레이블}} 딕셔너리 구성
code_map = {}
for _, row in df_cb.iterrows():
    orig_col  = str(row["변수코드"]).strip()
    label_val = str(row.get("코드레이블", "")).strip()
    code_val  = row["코드값"]

    if not label_val or label_val in ("nan", ""):
        continue
    if str(code_val).strip() in ("(연속형/자유값)", "(주관식)", "(없음)", "nan"):
        continue

    # 현재 df에서의 컬럼명 (rename 반영)
    cur_col = rename_map.get(orig_col, orig_col)

    if cur_col not in code_map:
        code_map[cur_col] = {}

    # 코드값을 숫자/문자 둘 다 키로 등록
    try:
        code_map[cur_col][int(float(code_val))] = label_val
    except (ValueError, TypeError):
        pass
    code_map[cur_col][str(code_val).strip()] = label_val

print(f"\n[코드북] 레이블 매핑 컬럼 수: {len(code_map)}개")

# ── 코드 치환 적용
replaced_log = []
for col, mapping in code_map.items():
    if col not in df.columns:
        continue

    # 텍스트 치환이 포함된 경우 dtype을 object로
    if pd.api.types.is_numeric_dtype(df[col]):
        df[col] = df[col].astype(object)

    before = df[col].copy()
    df[col] = df[col].map(lambda x: mapping.get(x, mapping.get(str(x).strip(), x)))

    changed = (before.astype(str) != df[col].astype(str)).sum()
    if changed > 0:
        replaced_log.append({"컬럼": col, "변환건수": changed,
                              "샘플": str(df[col].dropna().unique()[:3].tolist())})

print(f"  코드 치환 적용: {len(replaced_log)}개 컬럼")
for item in replaced_log:
    print(f"    {item['컬럼']:20s} {item['변환건수']:5,}건  →  {item['샘플']}")

# ══════════════════════════════════════════
# 4. 연도 컬럼 추가 & 최종 정리
# ══════════════════════════════════════════
df.insert(0, "연도", 2025)
print(f"\n[최종] {df.shape[0]:,}행 × {df.shape[1]}열")

# ══════════════════════════════════════════
# 5. 저장
# ══════════════════════════════════════════
out_csv  = OUTPUT_DIR / "survey_2025_standardized.csv"
out_xlsx = OUTPUT_DIR / "survey_2025_standardized.xlsx"

df.to_csv(out_csv, index=False, encoding="utf-8-sig")
df.to_excel(out_xlsx, index=False)

print(f"\n✅ 저장 완료")
print(f"   {out_csv}")
print(f"   {out_xlsx}")

# ── 간단한 결측률 요약
miss = df.isnull().mean().mul(100).round(1)
miss_high = miss[miss > 50].sort_values(ascending=False)
if len(miss_high):
    print(f"\n⚠️  결측률 50% 초과 컬럼 ({len(miss_high)}개):")
    print(miss_high.to_string())
else:
    print("\n✅ 결측률 50% 초과 컬럼 없음")

[원본] 9,000행 × 228열

[변수정의서]
  제거 대상: 0개  → []
  컬럼명 변경: 0개  → {}

[컬럼처리 후] 228열

[코드북] 레이블 매핑 컬럼 수: 6개
  코드 치환 적용: 0개 컬럼

[최종] 9,000행 × 229열

✅ 저장 완료
   ..\data\5_processed\survey_2025_standardized.csv
   ..\data\5_processed\survey_2025_standardized.xlsx

⚠️  결측률 50% 초과 컬럼 (145개):
Q2t5         100.0
Q20t12       100.0
Q24_1_9      100.0
Q24_1_8      100.0
Q24_1_7      100.0
Q24_1_6      100.0
Q24_1_5      100.0
Q22_2t       100.0
Q22_1t       100.0
Q21t11       100.0
Q4t5         100.0
Q24_1_11     100.0
Q18_2t       100.0
Q17t11       100.0
Q16_1t11     100.0
Q13_1t7      100.0
Q12t7        100.0
Q11t9        100.0
Q10t11       100.0
Q9_6t6       100.0
Q24_1_10     100.0
Q24_1_12     100.0
Q8_3_9t8     100.0
Q31_1_3      100.0
DQ2_2t6      100.0
Q31_4_1t7    100.0
Q31_4_A3t    100.0
Q31_4_A3     100.0
Q31_4_A2t    100.0
Q31_4_A1t    100.0
Q31_2t6      100.0
Q31_1_3t     100.0
Q31_1_2t     100.0
Q24_1_13     100.0
Q31_1_1t     100.0
Q28_1_3t     100.0
Q28_1_3      100.0
Q28_1_2t     10

# 2024